In [16]:
text = open("input.txt", 'r').read()

chars = list(sorted(set(text)))

vocab_size = len(chars)
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for i, s in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[n] for n in l])

In [61]:
import torch
batch_size = 32
block_size = 8



data = torch.tensor(encode(text), dtype=torch.long)
g = torch.Generator().manual_seed(1337)
n =int(0.9*len(text))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    split_data = train_data if split == 'train' else val_data
    ix = torch.randint(len(split_data)-block_size, (batch_size,), generator=g)
    x = torch.stack([split_data[i: i + block_size] for i in ix])
    y = torch.stack([split_data[i+1 : i + block_size +1] for i in ix])
    return x, y

xb, yb = get_batch('train')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t +1]
        target = yb[b, t]

In [62]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx)
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:,-1,:]
            probs = torch.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [63]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(loss)

print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

tensor(4.7313, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [65]:
import torch

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

for _ in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss)
    

tensor(2.5061, grad_fn=<NllLossBackward0>)


In [66]:

logits, loss = m(xb, yb)
print(loss)
#2.3159

print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))

tensor(2.5059, grad_fn=<NllLossBackward0>)


D:
paly lom, phe mou wid pt hath omabepan pielego le ad IUENTY:
Fis ran s, o ig rd.
che knd etndof orrea ch wo fe; it
nofes pimeistheer Bas tindshatr'd n ama rome; maren?
g aven yche byo-t h po
An! tht lt thabelinadorillais ste ithe a t fe bomey r tthape f gs om:
Y lt nrd ve far oto t hiereangou h 


In [67]:
xb, yb = get_batch('validation')
logits, loss = m(xb, yb)
print(loss)
# validation loss: 2.6305

tensor(2.6305, grad_fn=<NllLossBackward0>)


In [76]:
import torch
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn((B,T,C))
xbow = torch.zeros((B,T,C))

for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        print(xprev.shape)
        xbow[b, t] = xprev.mean(xprev, dim=0)

torch.Size([1, 2])


TypeError: mean() received an invalid combination of arguments - got (Tensor, dim=int), but expected one of:
 * (*, torch.dtype dtype = None)
      didn't match because some of the keywords were incorrect: dim
 * (tuple of ints dim, bool keepdim = False, *, torch.dtype dtype = None)
 * (tuple of names dim, bool keepdim = False, *, torch.dtype dtype = None)
